In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import types
from pyspark.sql import functions as F

gs_bucket = "gs://de-zoomcamp-2026-homework-07-bucket/"

cred_gcp = "/data/projects/data-engineering/data-engineering-zoomcamp-2026/homework/.key_gcs/de-zoomcamp-2026-486014-e58095ffb819.json"

spark = (
    SparkSession.builder
    .appName("zoomcamp-gcs")
    .config(
        "spark.jars.packages",
        "com.google.cloud.bigdataoss:gcs-connector:hadoop3-2.2.5"
    )
    .config("spark.hadoop.fs.gs.impl",
            "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl",
            "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true")
    .config("spark.hadoop.google.cloud.auth.service.account.json.keyfile", 
            cred_gcp)
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/14 14:47:49 WARN Utils: Your hostname, mballo-pc, resolves to a loopback address: 127.0.1.1; using 192.168.1.94 instead (on interface wlp1s0)
26/03/14 14:47:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/data/projects/data-engineering/data-engineering-zoomcamp-2026/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/mballo/.ivy2.5.2/cache
The jars for the packages stored in: /home/mballo/.ivy2.5.2/jars
com.google.cloud.bigdataoss#gcs-connector added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-db506d4b-eead-447c-9c54-549a38028470;1.0
	confs: [default]
	found com.google.cloud.bigdataoss#gcs-connector;hadoop3-2.2.5 in central
	found com.google.api-client#google-api-client-jackson2;1.32.2 in central
	found com.google

## Question 1: Install Spark and PySpark

- Install Spark
- Run PySpark
- Create a local spark session
- Execute spark.version.

What's the output?


In [2]:
spark.version

'4.1.1'

In [8]:
!gsutil ls gs://de-zoomcamp-2026-homework-07-bucket/

gs://de-zoomcamp-2026-homework-07-bucket/yellow_tripdata_2025-11.parquet
gs://de-zoomcamp-2026-homework-07-bucket/homework_M07/
gs://de-zoomcamp-2026-homework-07-bucket/pq/
gs://de-zoomcamp-2026-homework-07-bucket/raw/
gs://de-zoomcamp-2026-homework-07-bucket/report/


## Question 2: Yellow November 2025

Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

In [11]:
df = spark.read.parquet(f'{gs_bucket}homework_M07/yellow_tripdata_2025-11.parquet')
df.repartition(4).write.mode("overwrite").parquet(f'{gs_bucket}homework_M07/yellow_2025_11_repartitioned/')
!ls -lh gs_bucket/homework_M07/yellow_2025_11_repartitioned/*.parquet

In [14]:
!gsutil ls -lh gs://de-zoomcamp-2026-homework-07-bucket/homework_M07/yellow_2025_11_repartitioned/*.parquet

 24.41 MiB  2026-03-14T14:04:03Z  gs://de-zoomcamp-2026-homework-07-bucket/homework_M07/yellow_2025_11_repartitioned/part-00000-419b6ad9-664a-426d-8220-01f00c7f44c0-c000.snappy.parquet
 24.41 MiB  2026-03-14T14:04:03Z  gs://de-zoomcamp-2026-homework-07-bucket/homework_M07/yellow_2025_11_repartitioned/part-00001-419b6ad9-664a-426d-8220-01f00c7f44c0-c000.snappy.parquet
 24.42 MiB  2026-03-14T14:04:04Z  gs://de-zoomcamp-2026-homework-07-bucket/homework_M07/yellow_2025_11_repartitioned/part-00002-419b6ad9-664a-426d-8220-01f00c7f44c0-c000.snappy.parquet
 24.42 MiB  2026-03-14T14:04:04Z  gs://de-zoomcamp-2026-homework-07-bucket/homework_M07/yellow_2025_11_repartitioned/part-00003-419b6ad9-664a-426d-8220-01f00c7f44c0-c000.snappy.parquet
TOTAL: 4 objects, 102396945 bytes (97.65 MiB)


In [16]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



## Question 3: Count records

How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.


In [19]:
df.filter(F.to_date('tpep_pickup_datetime') == '2025-11-15').count()

162604

In [21]:
df.createOrReplaceTempView('trips_data_11_2025')

In [25]:
spark.sql(""" 
SELECT 
    count(*) as trips_count
FROM trips_data_11_2025
WHERE DATE(tpep_pickup_datetime) = '2025-11-15'

""").show()

[Stage 18:>                                                         (0 + 4) / 4]

+-----------+
|trips_count|
+-----------+
|     162604|
+-----------+



## Question 4: Longest trip

What is the length of the longest trip in the dataset in hours?


In [29]:
spark.sql("""
    SELECT 
        tpep_pickup_datetime,
        tpep_dropoff_datetime,
        (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600 AS duration_hours
    FROM trips_data_11_2025
    ORDER BY duration_hours DESC
    LIMIT 5
""").show()

[Stage 21:>                                                         (0 + 4) / 4]

+--------------------+---------------------+-----------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|   duration_hours|
+--------------------+---------------------+-----------------+
| 2025-11-26 20:22:12|  2025-11-30 15:01:00|90.64666666666666|
| 2025-11-27 04:22:41|  2025-11-30 09:19:35|76.94833333333334|
| 2025-11-03 10:42:55|  2025-11-06 14:55:45|76.21388888888889|
| 2025-11-07 11:23:22|  2025-11-10 08:40:41|69.28861111111111|
| 2025-11-18 17:12:47|  2025-11-21 12:17:37|67.08055555555555|
+--------------------+---------------------+-----------------+



In [30]:
df.withColumn(
    'duration_hours', (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600
) \
.orderBy('duration_hours', ascending=False) \
.select('tpep_pickup_datetime', 'tpep_dropoff_datetime', 'duration_hours') \
.show(5)

[Stage 22:==============>                                           (1 + 3) / 4]

+--------------------+---------------------+-----------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|   duration_hours|
+--------------------+---------------------+-----------------+
| 2025-11-26 20:22:12|  2025-11-30 15:01:00|90.64666666666666|
| 2025-11-27 04:22:41|  2025-11-30 09:19:35|76.94833333333334|
| 2025-11-03 10:42:55|  2025-11-06 14:55:45|76.21388888888889|
| 2025-11-07 11:23:22|  2025-11-10 08:40:41|69.28861111111111|
| 2025-11-18 17:12:47|  2025-11-21 12:17:37|67.08055555555555|
+--------------------+---------------------+-----------------+
only showing top 5 rows


## Question 6: Least frequent pickup location zone

Load the zone lookup data into a temp view in Spark:

```bash
wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
```

Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

In [31]:
!gsutil ls gs://de-zoomcamp-2026-homework-07-bucket/

gs://de-zoomcamp-2026-homework-07-bucket/homework_M07/
gs://de-zoomcamp-2026-homework-07-bucket/pq/
gs://de-zoomcamp-2026-homework-07-bucket/raw/
gs://de-zoomcamp-2026-homework-07-bucket/report/


In [40]:
df_zones = spark.read \
    .option('header', 'true') \
    .csv(f'{gs_bucket}homework_M07/zones/')

In [42]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [43]:
df_zones.createOrReplaceTempView('zones')

In [44]:
spark.sql("""
SELECT 
    COUNT(*) AS nbr_pickup,
    z.Zone AS Zone
    
FROM trips_data_11_2025 t 
JOIN zones z
    ON t.PULocationID=z.LocationID
GROUP BY z.Zone
ORDER BY nbr_pickup
""").show()

[Stage 27:=============================>                            (2 + 2) / 4]

+----------+--------------------+
|nbr_pickup|                Zone|
+----------+--------------------+
|         1|Governor's Island...|
|         1|Eltingville/Annad...|
|         1|       Arden Heights|
|         3|       Port Richmond|
|         4|       Rikers Island|
|         4|   Rossville/Woodrow|
|         4| Green-Wood Cemetery|
|         4|         Great Kills|
|         5|         Jamaica Bay|
|        12|         Westerleigh|
|        14|        Crotona Park|
|        14|             Oakwood|
|        14|New Dorp/Midland ...|
|        14|       West Brighton|
|        15|       Willets Point|
|        16|Breezy Point/Fort...|
|        17|Saint George/New ...|
|        18|       Broad Channel|
|        21|     Mariners Harbor|
|        22|Heartland Village...|
+----------+--------------------+
only showing top 20 rows


In [45]:
spark.sql("""
SELECT 
    COUNT(*) AS nbr_pickup,
    z.Borough AS Zone
    
FROM trips_data_11_2025 t 
JOIN zones z
    ON t.PULocationID=z.LocationID
GROUP BY z.Borough
ORDER BY nbr_pickup
""").show()

[Stage 31:>                                                         (0 + 4) / 4]

+----------+-------------+
|nbr_pickup|         Zone|
+----------+-------------+
|       392|Staten Island|
|       593|          EWR|
|      1897|          N/A|
|      6138|      Unknown|
|     34892|        Bronx|
|    154936|     Brooklyn|
|    374629|       Queens|
|   3607967|    Manhattan|
+----------+-------------+

